In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr
from scipy import stats
from scipy.spatial import cKDTree
from sklearn.linear_model import LinearRegression


In [2]:
data_dir = os.path.expanduser("~/DataFiles")
racmo_smb_file = data_dir + "/smbgl_monthlyS_ANT11_RACMO2.4p1_ERA5_197901_202512.nc"
racmo_t2m_file = data_dir + "/tas_monthlyA_ANT11_RACMO2.4p1_ERA5_197901_202512.nc"
racmo_mask_file = data_dir + "/TotIS_RACMO_ANT27_IMBIE2.nc"
ant11_mask_file = data_dir + "/ANT11_masks.nc"
ant27_ref_file  = data_dir + "/smb_monthlyS_ANT27_ERA5-3H_RACMO2.3p2_197901_202212.nc"
accum_data_file   = data_dir + "/AccumCoresData.csv"
water_data_file   = data_dir + "/WaterCoresData.csv"
accum_coords_file = data_dir + "/AccumCoresCoords.csv"
water_coords_file = data_dir + "/WaterCoresCoords.csv"


In [3]:
smb_ds = xr.open_dataset(racmo_smb_file)
smb_ann = (smb_ds["smbgl"].squeeze("height").groupby("time.year").sum("time").sel(year=slice(1979, 2025)))
t2m_ds = xr.open_dataset(racmo_t2m_file)
t2m_ann = (t2m_ds["tas"].squeeze("height").groupby("time.year").mean("time").sel(year=slice(1979, 2025)))
lat_racmo = smb_ann["lat"].values; lon_racmo = smb_ann["lon"].values
NY, NX = lat_racmo.shape
years_obs = smb_ann["year"].values
print("ANT11 grid:", (NY, NX), " years", int(years_obs.min()), "-", int(years_obs.max()))


ANT11 grid: (591, 726)  years 1979 - 2025


In [4]:
mask_ds = xr.open_dataset(ant11_mask_file)
cell_area_m2 = mask_ds["Area"].values * 1e6
ice_sheet    = mask_ds["Icesheet_Only"].values == 1

def to_xyz(lat, lon):
    la, lo = np.deg2rad(lat), np.deg2rad(lon)
    return np.stack([np.cos(la)*np.cos(lo), np.cos(la)*np.sin(lo), np.sin(la)], -1)

ref27 = xr.open_dataset(ant27_ref_file)["smb"].squeeze("height")
lat27, lon27 = ref27["lat"].values, ref27["lon"].values
gi = xr.open_dataset(racmo_mask_file)["GroundedIce"].values
gi = np.where(np.isfinite(gi), gi, 0).astype(int); _gm = gi > 0
_, _idx = cKDTree(to_xyz(lat27[_gm].ravel(), lon27[_gm].ravel())).query(to_xyz(lat_racmo.ravel(), lon_racmo.ravel()))
basin_mask = np.where(ice_sheet.ravel(), gi[_gm].ravel()[_idx], 0).reshape(NY, NX)

basin_flat  = basin_mask.reshape(-1); valid_cells = basin_flat > 0
basin_ids   = basin_flat[valid_cells].astype(int); basin_ids_unique = np.unique(basin_ids)
ais_mask  = basin_mask > 0
wais_mask = np.isin(basin_mask, [3, 7, 8, 9, 10])
eais_mask = np.isin(basin_mask, [1, 2, 11, 12, 13, 14, 15, 16, 17, 18])
ap_mask   = np.isin(basin_mask, [4, 5, 6])
masks = {"AIS": ais_mask, "WAIS": wais_mask, "EAIS": eais_mask, "AP": ap_mask}
region_names = ["AIS", "WAIS", "EAIS", "AP"]
print("grounded cells:", int(valid_cells.sum()))


grounded cells: 113482


In [5]:
def basin_sums_Gt(field3d, years):
    mass = field3d * cell_area_m2
    df = pd.DataFrame(index=list(np.asarray(years)), columns=basin_ids_unique, dtype=float)
    for t, yr in enumerate(np.asarray(years)):
        flat = mass[t].reshape(-1)[valid_cells]
        s = pd.DataFrame({"basin": basin_ids, "m": flat}).groupby("basin")["m"].sum() / 1e12
        df.loc[yr, s.index] = s.values
    return df


In [6]:
smb_basin_obs = basin_sums_Gt(smb_ann.values, years_obs)
ais_rac  = smb_basin_obs.sum(axis=1).astype(float)
wais_rac = smb_basin_obs[[3,7,8,9,10]].sum(axis=1).astype(float)
eais_rac = smb_basin_obs[[1,2,11,12,13,14,15,16,17,18]].sum(axis=1).astype(float)
ap_rac   = smb_basin_obs[[4,5,6]].sum(axis=1).astype(float)
raw_series = {"AIS": ais_rac, "WAIS": wais_rac, "EAIS": eais_rac, "AP": ap_rac}


In [7]:
accum_data_raw = pd.read_csv(accum_data_file, header=None)
water_data_raw = pd.read_csv(water_data_file, header=None)
def extract_cores(raw_df, yr_min, yr_max, transpose=True):
    years_col = raw_df.iloc[2:, 0].astype(int)
    m = (years_col >= yr_min) & (years_col <= yr_max)
    arr = raw_df.iloc[2:, 1:][m].to_numpy()[::-1].astype(float)
    return arr.T if transpose else arr

N_CAL = 42; CAL_YEARS = slice(1979, 2020); recon_years = np.arange(1801, 2001)
accum_cores_cal  = extract_cores(accum_data_raw, 1979, 2020)
water_cores_cal  = extract_cores(water_data_raw, 1979, 2020)
accum_cores_full = extract_cores(accum_data_raw, 1801, 2000, transpose=False)
water_cores_full = extract_cores(water_data_raw, 1801, 2000, transpose=False)

accum_coords = pd.read_csv(accum_coords_file, header=None)
water_coords = pd.read_csv(water_coords_file, header=None)
core_lats  = accum_coords.iloc[:, 2].values; core_lons  = (accum_coords.iloc[:, 3].values + 360) % 360
water_lats = water_coords.iloc[:, 2].values; water_lons = (water_coords.iloc[:, 3].values + 360) % 360
print("accum cores:", accum_cores_full.shape[1], " water cores:", water_cores_full.shape[1])


accum cores: 84  water cores: 80


In [8]:
def nearest_ij(lat2d, lon2d, tlat, tlon):
    lon360 = (lon2d + 360) % 360
    d2 = (lat2d - tlat)**2 + (lon360 - (tlon + 360) % 360)**2
    return np.unravel_index(np.argmin(d2), d2.shape)

smb_cal = smb_ann.sel(year=CAL_YEARS).values; t2m_cal = t2m_ann.sel(year=CAL_YEARS).values
accum_pts = np.zeros((84, N_CAL)); R_accum = np.zeros(84)
for i in range(84):
    j, k = nearest_ij(lat_racmo, lon_racmo, core_lats[i], core_lons[i])
    smb_pt = smb_cal[:, j, k]; obs = accum_cores_cal[i, :]; accum_pts[i, :] = smb_pt
    v = np.isfinite(obs) & np.isfinite(smb_pt)
    reg = LinearRegression().fit(smb_pt[v].reshape(-1,1), obs[v].reshape(-1,1))
    R_accum[i] = np.var(obs[v].reshape(-1,1) - reg.predict(smb_pt[v].reshape(-1,1)), ddof=1)
_valid = np.isfinite(accum_cores_cal) & np.isfinite(accum_pts)
_off = (np.nanmean(np.where(_valid, accum_cores_cal, np.nan), axis=1)
        - np.nanmean(np.where(_valid, accum_pts, np.nan), axis=1))
y_e_accum = accum_pts + _off[:, None]

y_e_water = np.zeros((80, N_CAL)); R_water = np.zeros(80)
for i in range(80):
    j, k = nearest_ij(lat_racmo, lon_racmo, water_lats[i], water_lons[i])
    t_pt = t2m_cal[:, j, k]; obs = water_cores_cal[i, :]
    v = np.isfinite(obs) & np.isfinite(t_pt)
    reg = LinearRegression().fit(t_pt[v].reshape(-1,1), obs[v].reshape(-1,1))
    y_e_water[i, :] = t_pt * reg.coef_[0][0] + reg.intercept_[0]
    R_water[i] = np.var(obs[v].reshape(-1,1) - reg.predict(t_pt[v].reshape(-1,1)), ddof=1)

y_e_smb    = np.vstack([y_e_accum, y_e_water])
R_smb_diag = np.concatenate([R_accum, R_water])
proxy_smb  = np.hstack([accum_cores_full, water_cores_full])
print("PSMs ready. y_e_smb", y_e_smb.shape)


PSMs ready. y_e_smb (164, 42)


In [9]:
def run_ensrf_spatial_unc(prior_cells, y_e, R_diag_vec, proxy_data_full, n_proxy, integ):
    Ns = prior_cells.shape[1]
    post_mean = np.zeros((200, Ns), np.float32); post_std = np.zeros((200, Ns), np.float32)
    reg_ens = np.zeros((200, N_CAL, integ.shape[0]))
    for i in range(200):
        X_mean = prior_cells.mean(axis=0); X_dev = prior_cells - X_mean
        for j in range(n_proxy):
            if np.isnan(proxy_data_full[i, j]): continue
            y_mean = y_e[j, :].mean(); y_var = np.var(y_e[j, :], ddof=1)
            if not np.isfinite(y_mean) or y_var == 0: continue
            y_dev = y_e[j, :] - y_mean
            K = ((X_dev.T @ y_dev) / (N_CAL - 1)) / (y_var + R_diag_vec[j])
            X_mean = X_mean + K * (proxy_data_full[i, j] - y_mean)
            alpha = 1.0 / (1.0 + np.sqrt(R_diag_vec[j] / (y_var + R_diag_vec[j])))
            X_dev = X_dev - alpha * np.outer(y_dev, K)
        post_mean[i] = X_mean; post_std[i] = X_dev.std(axis=0, ddof=1)
        reg_ens[i] = (integ @ (X_mean[None, :] + X_dev).T).T
    return post_mean, post_std, reg_ens


In [ ]:
_va = cell_area_m2.reshape(-1)[valid_cells]
_regions = {"AIS": basin_ids > 0, "WAIS": np.isin(basin_ids, [3,7,8,9,10]),
            "EAIS": np.isin(basin_ids, [1,2,11,12,13,14,15,16,17,18]), "AP": np.isin(basin_ids, [4,5,6])}
A_smb   = np.vstack([np.where(_regions[r], _va, 0) / 1e12 for r in region_names])          # (4, n_valid)
A_basin = np.vstack([np.where(basin_ids == b, _va, 0) / 1e12 for b in basin_ids_unique])   # (nb, n_valid)
A_all   = np.vstack([A_smb, A_basin])

smb_cells = smb_ann.sel(year=CAL_YEARS).values.reshape(N_CAL, -1)[:, valid_cells].astype(np.float32)
print("Running SMB EnSRF (%d grounded cells) ..." % smb_cells.shape[1])
smb_post_v, smb_std_v, reg_ens_all = run_ensrf_spatial_unc(smb_cells, y_e_smb, R_smb_diag, proxy_smb, 164, A_all)
smb_reg_ens   = reg_ens_all[:, :, :4]       # (200, N_CAL, 4)  regional ensemble
smb_basin_ens = reg_ens_all[:, :, 4:]       # (200, N_CAL, nb) basin ensemble

racmo_smb_recon = np.full((200, NY*NX), np.nan, np.float32); racmo_smb_recon[:, valid_cells] = smb_post_v
racmo_smb_recon = racmo_smb_recon.reshape(200, NY, NX)
print("done. AIS 1sigma (time-mean): %.1f Gt/yr" % smb_reg_ens[:, :, 0].std(axis=1, ddof=1).mean())


Running SMB EnSRF (113482 grounded cells) ...


In [ ]:
# ---- reduced (cell-free) EnSRF for any integration matrix; exact regional/basin posterior means ----
ymean = y_e_smb.mean(1); yvar = y_e_smb.var(1, ddof=1); ydev = y_e_smb - ymean[:, None]; Rd = R_smb_diag
avail = np.isfinite(proxy_smb)
def make_filter(A):
    Xm0 = smb_cells.mean(0); R0 = (smb_cells - Xm0) @ A.T; Jp = A @ Xm0; nreg = A.shape[0]
    def f(cols, years):
        cols = np.sort(np.asarray(cols)); R_ens = R0.copy(); G = np.empty((cols.size, nreg))
        for m, j in enumerate(cols):
            den = yvar[j] + Rd[j]
            if not np.isfinite(den) or den == 0: G[m] = 0.0; continue
            g = (R_ens.T @ ydev[j]) / (N_CAL - 1) / den
            R_ens -= (1.0/(1.0+np.sqrt(Rd[j]/den))) * np.outer(ydev[j], g); G[m] = g
        return Jp + (proxy_smb[np.ix_(years, cols)] - ymean[cols]) @ G
    return f

# ---- King Eqs (6)-(7): frozen-network variance ratio per basin -> per-year weights ----
fb = make_filter(A_basin); nb = A_basin.shape[0]; MINYRS = 10
base = fb(np.where(avail.all(0))[0], np.arange(200))
uniq, inv = np.unique(avail, axis=0, return_inverse=True)
P = np.full((uniq.shape[0], nb), np.nan)
for u in range(uniq.shape[0]):
    cols = np.where(uniq[u])[0]; yrs = np.where(avail[:, cols].all(1))[0]
    if yrs.size < MINYRS or cols.size == 0: continue
    P[u] = fb(cols, yrs).std(0, ddof=1) / base[yrs].std(0, ddof=1)
w_b = P[inv] / np.nanmax(P, axis=0); w_b[~np.isfinite(w_b)] = 1.0     # (200, nb)
print("basins:", nb, " skipped patterns:", int(np.isnan(P).any(1).sum()), "/", uniq.shape[0])

# ---- apply /w to the basin ensemble; aggregate to regions ----
base_b = smb_basin_ens[:100].mean(axis=(0, 1))
smb_basin_ens = base_b + (smb_basin_ens - base_b) / w_b[:, None, :]
bcol_of = {b: i for i, b in enumerate(basin_ids_unique)}
reg_cols = {"AIS":  [bcol_of[b] for b in basin_ids_unique],
            "WAIS": [bcol_of[b] for b in [3,7,8,9,10] if b in bcol_of],
            "EAIS": [bcol_of[b] for b in [1,2,11,12,13,14,15,16,17,18] if b in bcol_of],
            "AP":   [bcol_of[b] for b in [4,5,6] if b in bcol_of]}
smb_reg_ens  = np.stack([smb_basin_ens[:, :, reg_cols[r]].sum(2) for r in region_names], axis=2)
smb_reg_mean = smb_reg_ens.mean(1); smb_reg_std = smb_reg_ens.std(1, ddof=1)
ais_rac_recon, wais_rac_recon, eais_rac_recon, ap_rac_recon = (smb_reg_mean[:, k] for k in range(4))
ais_rac_std,  wais_rac_std,  eais_rac_std,  ap_rac_std      = (smb_reg_std[:, k]  for k in range(4))
smb_std_by_region = {"AIS": ais_rac_std, "WAIS": wais_rac_std, "EAIS": eais_rac_std, "AP": ap_rac_std}
recon_series = {"AIS": ais_rac_recon, "WAIS": wais_rac_recon, "EAIS": eais_rac_recon, "AP": ap_rac_recon}

# ---- apply /w to the grid cube (per cell by its basin) ----
bcell = np.full((NY, NX), -1, int)
for i, b in enumerate(basin_ids_unique): bcell[basin_mask == b] = i
valid2d = bcell >= 0
cellmean = np.nanmean(racmo_smb_recon[:100], axis=0)
wcell = w_b[:, np.where(valid2d, bcell, 0)]; wcell[:, ~valid2d] = 1.0
racmo_smb_recon = cellmean[None] + (racmo_smb_recon - cellmean[None]) / wcell     # variance-corrected cube
smb_basin_recon = basin_sums_Gt(racmo_smb_recon, recon_years)
print("Variance-corrected series installed.")


In [ ]:
def _ggm_psi(kappa, phi, K):
    d = 0.5*kappa; a = 1.0-phi; psi = np.empty(K); psi[0] = 1.0
    for k in range(1, K): psi[k] = psi[k-1]*(d+k-1.0)/k*a
    return psi
def _ggm_acf(kappa, phi, n, K=None):
    if K is None: K = int(min(20000, max(2000, 30*n if phi > 0.02 else 20000)))
    psi = _ggm_psi(kappa, phi, K+n)
    return np.array([np.dot(psi[:K], psi[h:h+K]) for h in range(n)])
def _toeplitz(r):
    n = len(r); idx = np.abs(np.arange(n)[:, None] - np.arange(n)[None, :]); return r[idx]
def ggm_fit(x, kappa0=1.0, phi0=0.1, bounds=((0.0,4.0),(1e-4,1.0))):
    from scipy.optimize import minimize
    x = np.asarray(x, float); x = x - x.mean(); n = len(x)
    def negll(th):
        r = _ggm_acf(th[0], th[1], n)
        try: L = np.linalg.cholesky(_toeplitz(r))
        except np.linalg.LinAlgError: return 1e12
        z = np.linalg.solve(L, x); s2 = (z@z)/n
        return 0.5*(n*np.log(2*np.pi*s2) + 2.0*np.log(np.diag(L)).sum() + n)
    res = minimize(negll, [kappa0, phi0], method="L-BFGS-B", bounds=bounds)
    r = _ggm_acf(*res.x, n); L = np.linalg.cholesky(_toeplitz(r)); z = np.linalg.solve(L, x)
    return dict(kappa=res.x[0], phi=res.x[1], sigma_w=np.sqrt((z@z)/n))

PI = slice(0, 100)
ggm_fits = {r: ggm_fit(np.asarray(recon_series[r])[PI]) for r in region_names}
for r in region_names:
    f = ggm_fits[r]; print(f"{r}: kappa {f['kappa']:.2f}  phi {f['phi']:.3f}  sigma_w {f['sigma_w']:.1f}")
=